![electronic_medical_records](electronic_medical_records.png)

Medical professionals often summarize patient encounters in transcripts written in natural language, which include details about symptoms, diagnosis, and treatments. These transcripts can be used for other medical documentation, such as for insurance purposes, but as they are densely packed with medical information, extracting the key data accurately can be challenging.  

You and your team at Lakeside Healthcare Network have decided to leverage the OpenAI API to automatically extract medical information from these transcripts and automate the matching with the appropriate ICD-10 codes. ICD-10 codes are a standardized system used worldwide for diagnosing and billing purposes, such as insurance claims processing.

## The Data
The dataset contains anonymized medical transcriptions categorized by specialty.

## transcriptions.csv
| Column     | Description              |
|------------|--------------------------|
| `"medical_specialty"` | The medical specialty associated with each transcription.  |
| `"transcription"` | Detailed medical transcription texts, with insights into the medical case. |

In [18]:
# Import the necessary libraries
import pandas as pd
from openai import OpenAI
import json

In [19]:
# Load the data
df = pd.read_csv("data/transcriptions.csv")
df.head()

,medical_specialty,transcription
0,Allergy / Immunology,"SUBJECTIVE:, This 23-year-old white female pr..."
1,Orthopedic,"CHIEF COMPLAINT:, Achilles ruptured tendon.,H..."
2,Bariatrics,"PREOPERATIVE DIAGNOSIS: , Morbid obesity.,POST..."
3,Cardiovascular / Pulmonary,"PREOPERATIVE DIAGNOSES,Airway obstruction seco..."
4,Urology,"CHIEF COMPLAINT:, Urinary retention.,HISTORY ..."


In [20]:
# Initialize the OpenAI client
client = OpenAI()

# Define function to extract age and recommended treatment/procedure
def extract_info_with_openai(transcription):
    """Extracts age and recommended treatment from a transcription using OpenAI."""
    messages = [
        {
            "role": "system",
            "content": "You are a healthcare professional extracting patient data. Always return both the age and recommended treatment. If the information is missing, still create the field and specify 'Unknown'.",
            "role": "user",
            "content": f"Please extract and return both the patient's age and recommended treatment from the following transcription. Transcription: {transcription}."
        }
    ]
    function_definition = [
        {
            'type': 'function',
            'function': {
                'name': 'extract_medical_data',
                'description': 'Get the age and recommended treatment from the input text. Always return both age and recommended treatment.',
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'Age': {
                            'type': 'integer',
                            'description': 'Age of the patient'
                        },
                        'Recommended Treatment/Procedure': {
                            'type': 'string',
                            'description': 'Recommended treatment or procedure for the patient'
                        }
                    }
                }
            }
        }
    ]
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=function_definition
    )
    return json.loads(response.choices[0].message.tool_calls[0].function.arguments)

# Define function to extract age and recommended treatment/procedure
def get_icd_codes(treatment):
    if treatment != 'Unknown':
        """Retrieves ICD codes for a given treatment using OpenAI."""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": f"Provide the ICD codes for the following treatment or procedure: {treatment}. Return the answer as a list of codes. Please only include the codes and no other information."
            }],
            temperature=0.2
        )
        output = response.choices[0].message.content
    else:
        output = 'Unknown'
    return output

# Start an empty list to store processed data
processed_data = []

# Process each row in the DataFrame
for index, row in df.iterrows():
    medical_specialty = row['medical_specialty']
    extracted_data = extract_info_with_openai(row['transcription'])
    icd_code = get_icd_codes(extracted_data["Recommended Treatment/Procedure"]) if 'Recommended Treatment/Procedure' in extracted_data.keys() else 'Unknown'
    extracted_data["Medical Specialty"] = medical_specialty
    extracted_data["ICD Code"] = icd_code

    # Append the extracted information as a new row in the list
    processed_data.append(extracted_data)

# Convert the list to a DataFrame
df_structured = pd.DataFrame(processed_data)


In [23]:
df_structured.head()

,Age,Recommended Treatment/Procedure,Medical Specialty,ICD Code
0,23,"Try Zyrtec instead of Allegra, loratadine as a...",Allergy / Immunology,- Zyrtec: J30.9\n- Allegra: J30.9\n- Loratadin...
1,41,operative fixation,Orthopedic,- 81.00\n- 81.01\n- 81.02\n- 81.03\n- 81.04\n-...
2,30,Laparoscopic antecolic antegastric Roux-en-Y g...,Bariatrics,- 43644\n- 43645\n- 43647
3,50,Neck exploration; tracheostomy; urgent flexibl...,Cardiovascular / Pulmonary,- 0C9B0ZZ\n- 0B110F4\n- 0B110F4\n- 0B110F4\n- ...
4,66,self-catheterize if urinary retention occurs a...,Urology,- N39.3\n- Z79.899\n- Z79.1


In [ ]:
# ================================
# 🏥 Medical Data Extraction Pipeline (All-in-One)
# ================================

import pandas as pd
import json
import logging
from openai import OpenAI
from tqdm import tqdm

# -------------------------------
# ⚙️ Config
# -------------------------------
# API_KEY = "YOUR_API_KEY"  # Set your OpenAI API key here
INPUT_FILE = "data/transcriptions.csv"
OUTPUT_FILE = "structured_output.csv"

# -------------------------------
# 🔧 Setup
# -------------------------------
client = OpenAI()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# -------------------------------
# 🧠 Prompt Template
# -------------------------------
SYSTEM_PROMPT = """
You are a medical data extraction system.

Extract the following:
- age (number or null)
- recommended_treatment (string or null)

Return ONLY valid JSON in this format:
{
  "age": int | null,
  "recommended_treatment": string | null
}
"""

# -------------------------------
# ✅ Validation Function
# -------------------------------
def validate_output(response_text):
    try:
        data = json.loads(response_text)

        return {
            "age": data.get("age", None),
            "recommended_treatment": data.get("recommended_treatment", None)
        }
    except:
        return {
            "age": None,
            "recommended_treatment": None
        }

# -------------------------------
# 🚀 Extraction Function
# -------------------------------
def extract_info(transcription):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": transcription}
            ],
            temperature=0
        )

        content = response.choices[0].message.content.strip()
        return validate_output(content)

    except Exception as e:
        logging.error(f"Error: {e}")
        return {
            "age": None,
            "recommended_treatment": None
        }

# -------------------------------
# 📂 Load Data
# -------------------------------
try:
    df = pd.read_csv(INPUT_FILE)
    assert "transcription" in df.columns
except Exception as e:
    raise Exception("CSV must contain 'transcription' column")

# -------------------------------
# 🔁 Process Data
# -------------------------------
logging.info("🚀 Processing started...")

results = []
for text in tqdm(df["transcription"], desc="Processing"):
    results.append(extract_info(text))

# -------------------------------
# 📊 Merge Results
# -------------------------------
structured_df = pd.DataFrame(results)
final_df = pd.concat([df, structured_df], axis=1)

# -------------------------------
# 💾 Save Output
# -------------------------------
final_df.to_csv(OUTPUT_FILE, index=False)

logging.info("✅ Processing completed!")
logging.info(f"📁 Saved to {OUTPUT_FILE}")

Processing: 100%|██████████| 5/5 [00:10<00:00,  2.10s/it]


In [3]:
structured_df

,age,recommended_treatment
0,23,Try Zyrtec instead of Allegra; another option ...
1,41,operative fixation
2,30,Laparoscopic antecolic antegastric Roux-en-Y g...
3,50,Neck exploration; tracheostomy; urgent flexibl...
4,66,self-catheterization in the event of urinary r...
